# HW2 - Part 2 Template Attack
## Eyal Rechef - 213099468, Alon Abudraham - 324231992

### 1. Modifying the C code

Basically for our template attack we would want to change keys frequently, so we'll have to add a command to switch the key frequently,

For maximum flexibility, we'll just use one command which receives plaintext and a key (32 bytes) and returns the ciphertext (16 bytes), the C code:

```C
#include "aes-independant.h"
#include "hal.h"
#include "simpleserial.h"
#include <stdint.h>
#include <stdlib.h>

uint8_t get_pt(uint8_t *pt, uint8_t len)
{
	/*
	 * get_pt - does AES block encryption
	 *
	 * receives as input both key and a plaintext
	 * 'p' [16 bytes of plaintext] [16 bytes of key]
	 * 
	 * This function is given as a callback to the simpleserial interface
	 * will be called after receiving a command 'p' and a 32 byte input
	 */
	aes_indep_enc_pretrigger(pt);
	aes_indep_init();
	aes_indep_key(&pt[KEY_LENGTH]);

	trigger_high();
	aes_indep_enc(pt); /* encrypting the data block */
	trigger_low();

	aes_indep_enc_posttrigger(pt);

	simpleserial_put('r', KEY_LENGTH, pt);
	return 0x00;
}

int main(void)
{
	uint8_t secret_key[KEY_LENGTH] = {DEFAULT_KEY};

	platform_init();
	init_uart();
	trigger_setup();

	aes_indep_init();
	aes_indep_key(secret_key); // sets the key. size = 16 bytes

	simpleserial_init();
	simpleserial_addcmd('p', 2 * KEY_LENGTH, get_pt);
	while (1)
		simpleserial_get();
}
```


### Compiling our C code
Our C firmware for the STM target is found in this directory under the name `chipwhisperer_hw2_template.c`

First we will just compile the C firwmare for the target, there is a simple bash script here which calls the `makefile`:

In [1]:
%%bash
make PLATFORM=CWLITEARM SOURCE=chipwhisperer_hw2_template.c

No CRYPTO_TARGET passed - defaulting to TINYAES128C
Building for platform CWLITEARM with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Compiling:
-en     chipwhisperer_hw2_template.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware/mcu/simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware/mcu/hal/hal.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware/mcu/hal//stm32f3/stm32f3_hal.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware/mcu/hal//stm32f3/stm32f3_hal_lowlevel.c ...
-e Done!
.
Compiling:
-en     ../../../../firmware

## Simple imports and setting up the CW handler
We've added a simple object to make life easier for us: 

In [2]:
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange


class CWHandler:
    DEFAULT_FIRMWARE_HEX = "output-CWLITEARM.hex"
    TARGET_CMD_OPCODE = "p"
    AES_BLOCK_SIZE = 128
    AES_BLOCK_SIZE_BYTES = AES_BLOCK_SIZE // 8

    def __init__(self, adc_samples_num: int):
        """
        Initializes a CWHandler object
        Args:
            adc_samples_num (int): Number of sampmles within a
            trace outputed by the Chip Whisperer
        """
        self.scope = cw.scope()
        self.target = cw.target(self.scope, cw.targets.SimpleSerial)
        self.scope.default_setup()
        self.scope.adc.samples = adc_samples_num
        
    def __del__(self):
        self.scope.dis()
        self.target.dis()

    def program_target(self, hex_file: str):
        """
        Programs the STM32 target on the Chip Whisperer
        Args:
            hex_file (str): hex_file
        """
        cw.program_target(self.scope, cw.programmers.STM32FProgrammer, hex_file)

    def program_with_our_code(self):
        self.program_target(CWHandler.DEFAULT_FIRMWARE_HEX)

    def send_command(self, opcode: str, payload: bytes):
        self.target.flush()
        self.target.simpleserial_write(opcode, payload)
        ret = self.scope.capture()
        if ret:
            raise Exception("Error in capturing response from command")
        return ret

    def encrypt_and_capture_trace(self, plaintext: bytes, key: bytes) -> np.ndarray:
        self.target.flush()
        self.scope.arm()

        assert len(plaintext) == CWHandler.AES_BLOCK_SIZE_BYTES
        assert len(key) == CWHandler.AES_BLOCK_SIZE_BYTES

        self.target.simpleserial_write(CWHandler.TARGET_CMD_OPCODE, plaintext + key)

        ret = self.scope.capture()
        if ret:
            raise Exception("Error in capturing response from command")
        
        trace = self.scope.get_last_trace()
        response = self.target.simpleserial_read('r', CWHandler.AES_BLOCK_SIZE_BYTES)
        return response, trace

c:\Users\eyalr\ChipWhisperer\chipwhisperer\jupyter\.venv\Lib\site-packages\chipwhisperer\capture\trace\TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


Let's create a instance of our `CWHandler` - and program the target

In [3]:
ADC_SAMPLE_NUM = 1500

handler = CWHandler(ADC_SAMPLE_NUM)

handler.program_with_our_code()


scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 2160507                   to 15695314                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.clkgen_div                   changed from 1                         to 26                       
scope.clock.clkgen_freq                  changed from 192000000.0               to 7384615.384615385        
scope.io.tio1      

### 2. Developing the strategy

Before starting with our template attack, let's feel the water.

We'll run a simple test -

- First we'll create a function that receives a key, and outputs an averaged power trace of the AES operation with that key, together with a plaintext of 0s

- Next will want to see the impact of difference in each of the keys bytes, so will send a key that is 0s, apart from the byte we are currently checking, this byte will once be 0x00, and once be 0xFF

- We'll plot all the power traces to see if we are in the right direction - does it make any sense

In [4]:
def get_average_trace_for_key(key: bytes, trace_num: int) -> np.ndarray:
    traces = []

    for i in range(trace_num):
        _, trace = handler.encrypt_and_capture_trace(
            b"\x00" * CWHandler.AES_BLOCK_SIZE_BYTES, key
        )
        traces.append(trace)

    res = np.array([np.float64(0)] * ADC_SAMPLE_NUM)

    for trace in traces:
        res += trace

    return np.abs(res / trace_num)


SAMPLE_SIZE = 10


def sample_test(default_byte: int, changed_byte: int):
    rows = int(np.sqrt(CWHandler.AES_BLOCK_SIZE_BYTES))
    fig, axes = plt.subplots(rows, rows, figsize=(20, 20))

    for i in range(CWHandler.AES_BLOCK_SIZE_BYTES):
        key = bytearray(
            (default_byte.to_bytes(1, "big")) * CWHandler.AES_BLOCK_SIZE_BYTES
        )
        avg_trace_key_0s = get_average_trace_for_key(key, SAMPLE_SIZE)
        key[i] = changed_byte
        avg_trace_key_1s = get_average_trace_for_key(key, SAMPLE_SIZE)

        ax = axes[i // rows][i % rows]
        ax.plot(avg_trace_key_0s, alpha=0.5)
        ax.plot(avg_trace_key_1s, alpha=0.5)
        ax.set_xlabel("sample num")
        ax.set_ylabel("ADC sample [V]")
        ax.set_title(f"Key byte #{i}")

    plt.show()


# sample_test(0x0, 0xFF)

The difference is very celar, with incrementing the key, we can see different power peaks changing between the case in which the byte is 0xFF (orange), and between the case the byte is 0x00 (blue).

Let's try to do something else, let's now try to change the default byte to 0xF, and each time change one byte to 0xFF, since the HW difference is now 4 and not 8, we would expect to see smaller changes in the peaks:

In [5]:
# sample_test(0x0F, 0xFF)

Well we indeed see smaller diffs.

And now, let's try value for the changed byte, with the same HW (say 0xF0), and now we would expect to see that the difference is minimal:

In [6]:
# sample_test(0x0F, 0xF0)

It seems as though there is no difference now, makes sense up until now.

From now, it seems that there is 1 POI for each byte key, let's define our method of creating the template.

We'll first find our POIs mathematically, and not by "eye".

For each byte, we'll sweep the 256 byte options (when the rest of the bytes are constant 0x0) for each one we'll then calculate

Referring to the formula from class:

$$
M_{k, i} = \frac{1}{T_k}\sum_{j=1}^{T_k}t_{j,i}
$$

In our case $T_k=5$, $k$ is the byte sweeping from $0$ to $255$.

Then we'll calculate:

$$
D_i = \sum_{k_1, k_2}\left|M_{k_1, i} - M_{k_2, i}\right|
$$

Then our POI will be the point with the highest $D_i$.

Then when we'll have all $D$ - we can calculate the POIs by finding the highest point of $D_i$.
To find few POIs:

- Pick the highest point in $D_i$ and save this as the $n$-th POI
- Throw out the nearest $N$ points
- Repeat until enough POIs have been selected

We'll start with 3 POIs

In [ ]:
TK = 5
MIN_POI_SPACING = 25

NUM_OF_POIS = 3

BYTE_OPTIONS = 2**8


def get_d_for_byte_idx(byte_idx: int, samples_num: int) -> np.ndarray:
    m_k = []
    for byte in range(BYTE_OPTIONS):
        key = bytearray(b"\x00" * CWHandler.AES_BLOCK_SIZE_BYTES)
        key[byte_idx] = byte
        m_k.append(get_average_trace_for_key(key, samples_num))

    d = np.array([np.float64(0)] * ADC_SAMPLE_NUM)
    for k1 in range(BYTE_OPTIONS):
        for k2 in range(BYTE_OPTIONS):
            d += np.abs(m_k[k1] - m_k[k2])

    return d


def extract_poi_from_d_vector(d: np.ndarray, num_of_pois: int, min_poi_spacing: int):
    poi_vector = []
    for _ in range(num_of_pois):
        poi_vector.append(np.argmax(d))
        d = np.delete(
            d, slice(poi_vector[-1] - min_poi_spacing, poi_vector[-1] + min_poi_spacing)
        )

    return poi_vector


# Get all d_i vector
pois_array = []
# for idx in range(CWHandler.AES_BLOCK_SIZE_BYTES):
for idx in range(1):
    curr_d = get_d_for_byte_idx(idx, TK)
    pois = extract_poi_from_d_vector(curr_d, NUM_OF_POIS, MIN_POI_SPACING)
    pois_array.append(pois)
    print(f"Our POI for key byte #{idx} is {pois}")

Our POI for key byte #0 is [1321, 633, 1249]


KeyboardInterrupt: 

Now let's try to create a template, we have a POI for each 

Deleting `handler` - disconnects us from the ChipWhisperer

In [ ]:
del handler